<a href="https://githubtocolab.com/eskayML/issr_gsoc2026_communication_analysis/blob/main/02_Audio_Enhancement.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install dependencies if running in Colab
import sys
if 'google.colab' in sys.modules:
    !pip install librosa==0.11.0 noisereduce==3.0.3 soundfile==0.12.1

# GSoC 2026: ISSR Project 01 Test
## Notebook 2: Audio Enhancement Algorithm & Method Evaluation

**Applicant:** Samuel Kalu

### 1. Tailoring Enhancement for the 6+1 Participant Simulator
Based on my understanding of the ISSR driving simulator, we face a high density of overlapping speech from **6 participants plus a narrator**, compounded by the mechanical drone of the simulator itself. 

To address this, my pipeline doesn't just reduce noise; it aims to **reconstruct clarity**. By using a pre-emphasis filter before denoising, we preserve the high-frequency formants (2-4kHz) that are essential for differentiating between multiple simultaneous speakers.

**The Multi-Stage Pipeline:**
1.  **Pre-emphasis:** Sharpens formants to help separate 7 distinct voices.
2.  **Spectral Gating:** Dynamically subtracts the simulator's stationary noise profile.
3.  **Peak Normalization:** Prepares the audio for high-accuracy transcription (Whisper).

In [ ]:
import os
import librosa
import librosa.display
import numpy as np
import soundfile as sf
import noisereduce as nr
import matplotlib.pyplot as plt
from scipy.signal import lfilter

class AudioProcessor:
    def __init__(self, sample_rate=16000):
        self.sr = sample_rate

    def load_audio(self, file_path):
        y, sr = librosa.load(file_path, sr=self.sr)
        return y, sr

    def pre_emphasis_filter(self, y, coeff=0.97):
        # Crucial for the 7-participant (6+1) clarity
        return lfilter([1, -coeff], [1], y)

    def spectral_gating_denoise(self, y, stationary=False, prop_decrease=1.0):
        return nr.reduce_noise(y=y, sr=self.sr, stationary=stationary, prop_decrease=prop_decrease)

    def normalize(self, y, target_db=-0.1):
        max_val = np.max(np.abs(y))
        if max_val > 0:
            target_amplitude = 10**(target_db / 20)
            return y * (target_amplitude / max_val)
        return y

    def process_pipeline(self, input_path, output_path):
        y, sr = self.load_audio(input_path)
        y_clarified = self.pre_emphasis_filter(y)
        y_denoised = self.spectral_gating_denoise(y_clarified)
        y_final = self.normalize(y_denoised)
        sf.write(output_path, y_final, self.sr)
        return y_final, self.sr

def calculate_snr(signal, noise):
    signal_power = np.mean(signal**2)
    noise_power = np.mean(noise**2)
    if noise_power == 0: return float('inf')
    return 10 * np.log10(signal_power / noise_power)

# 1. Load the actual AMI reference (Downloaded in Notebook 1)
input_path = 'input/AMI_sample.wav'
if not os.path.exists(input_path):
    print("Error: input/AMI_sample.wav not found. Please run Notebook 1 first.")
else:
    processor = AudioProcessor()
    y_clean, sr = processor.load_audio(input_path)

    # 2. Simulate Noisy ISSR Simulator Environment
    np.random.seed(42)
    # Noise tailored to simulator characteristics (hum + wideband noise)
    noise = 0.01 * np.random.randn(len(y_clean)) + 0.03 * np.sin(2 * np.pi * 60 * np.arange(len(y_clean)) / sr)
    y_noisy = y_clean + noise

    noisy_path = 'input/input_noisy_sample.wav'
    sf.write(noisy_path, y_noisy, sr)
    print(f"Saved noisy simulator audio to {noisy_path}")


### 2. Executing the Enhancement Pipeline
We use the `AudioProcessor` to run the full pipeline. This simulates the automated batch processing I will build for the final GSoC tool.

In [ ]:
enhanced_path = 'output/output_enhanced_sample.wav'
print("Running enhancement pipeline...")
y_enhanced, _ = processor.process_pipeline(noisy_path, enhanced_path)
print(f"Enhanced audio saved to {enhanced_path}")


### 3. Evaluation and Showcase Analysis
Comparative visualizations and quantitative SNR analysis.

In [ ]:
def plot_comparison(original, noisy, enhanced, sr):
    plt.figure(figsize=(15, 12))
    # Clean (subset for visibility)
    plt.subplot(3, 2, 1); librosa.display.waveshow(original[:160000], sr=sr); plt.title('Clean Signal Waveform')
    plt.subplot(3, 2, 2); D_clean = librosa.amplitude_to_db(np.abs(librosa.stft(original[:160000])), ref=np.max)
    librosa.display.specshow(D_clean, sr=sr, x_axis='time', y_axis='hz'); plt.title('Clean Spectrogram')
    # Noisy
    plt.subplot(3, 2, 3); librosa.display.waveshow(noisy[:160000], sr=sr); plt.title('Noisy Simulator Signal (Input)')
    plt.subplot(3, 2, 4); D_noisy = librosa.amplitude_to_db(np.abs(librosa.stft(noisy[:160000])), ref=np.max)
    librosa.display.specshow(D_noisy, sr=sr, x_axis='time', y_axis='hz'); plt.title('Noisy Spectrogram')
    # Enhanced
    plt.subplot(3, 2, 5); librosa.display.waveshow(enhanced[:160000], sr=sr); plt.title('Enhanced Signal (Output)')
    plt.subplot(3, 2, 6); D_enhanced = librosa.amplitude_to_db(np.abs(librosa.stft(enhanced[:160000])), ref=np.max)
    librosa.display.specshow(D_enhanced, sr=sr, x_axis='time', y_axis='hz'); plt.title('Enhanced Spectrogram')
    plt.tight_layout(); plt.show()

plot_comparison(y_clean, y_noisy, y_enhanced, sr)

snr_before = calculate_snr(y_clean, noise)
residual_noise = y_enhanced - y_clean
snr_after = calculate_snr(y_clean, residual_noise)

print(f"--- Quantitative Analysis ---")
print(f"SNR Before Enhancement: {snr_before:.2f} dB")
print(f"SNR After Enhancement:  {snr_after:.2f} dB")
print(f"Net SNR Improvement:    {snr_after - snr_before:.2f} dB")
